# Mean Reversion Experiment

Testing whether extreme 3-day returns tend to revert in the next day.

**Hypothesis**: Large moves (positive or negative) over 3 days should be followed by opposite moves in the next day (mean reversion).

In [ ]:
# Setup imports - now works automatically after pip install -e .
from pathlib import Path
import pandas as pd

from mini_research_lab import (
    download_prices,
    add_return_features,
    MiniResearchLab,
    default_experiments,
    plot_experiment_bundle,
)

In [ ]:
# Download and prepare data
prices = download_prices("AAPL", start="2018-01-01")
df = add_return_features(prices)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
# Set up experiment
lab = MiniResearchLab(df)
spec = default_experiments()[0]  # mean_reversion_3d_to_1d

print(f"Experiment: {spec.name}")
print(f"Description: {spec.description}")
print(f"X variable: {spec.x_col}")
print(f"Y variable: {spec.y_col}")

In [ ]:
# Run experiment
result = lab.run_experiment(spec.x_col, spec.y_col)

# Display results
print("=== X Variable Summary ===")
print(pd.Series(result["x_describe"].to_dict()))

print("\n=== Y Variable Summary ===")
print(pd.Series(result["y_describe"].to_dict()))

In [ ]:
# Regression results
print("=== Regression Summary ===")
regression_df = pd.DataFrame([result["regression"].to_dict()]).T
regression_df.columns = ['value']
print(regression_df)

print("\n=== Full Model Summary ===")
print(result["model"].summary())

In [ ]:
# Generate and save plots to spec folder in shared reports/figures
figures_dir = Path("../reports/figures") / spec.name
saved_files = plot_experiment_bundle(
    df, spec.x_col, spec.y_col, 
    title_prefix="",  # No prefix since we're using folder structure
    output_dir=figures_dir
)

print("Saved plots:")
for file_path in saved_files:
    print(f"  {file_path}")

In [ ]:
# Save all summaries to spec folder in shared reports/tables
tables_dir = Path("../reports/tables") / spec.name
tables_dir.mkdir(parents=True, exist_ok=True)

# Save X summary to separate file
with open(tables_dir / "x_summary.txt", "w") as f:
    f.write("=== X Variable Summary ===\n")
    x_dict = result["x_describe"].to_dict()
    for key, value in x_dict.items():
        f.write(f"{key}: {value}\n")

# Save Y summary to separate file
with open(tables_dir / "y_summary.txt", "w") as f:
    f.write("=== Y Variable Summary ===\n")
    y_dict = result["y_describe"].to_dict()
    for key, value in y_dict.items():
        f.write(f"{key}: {value}\n")

# Save regression summary to separate file
with open(tables_dir / "regression.txt", "w") as f:
    f.write("=== Regression Summary ===\n")
    reg_dict = result["regression"].to_dict()
    for key, value in reg_dict.items():
        f.write(f"{key}: {value}\n")

# Save full model summary as text
with open(tables_dir / "model_summary.txt", "w") as f:
    f.write(str(result["model"].summary()))

print(f"Results saved to:")
print(f"  Tables: {tables_dir}")
print(f"  Figures: {figures_dir}")

## Interpretation

### Key Results:
- **Coefficient**: Shows the relationship between 3-day returns and next-day returns
- **P-value**: Indicates statistical significance (< 0.05 is significant)
- **R-squared**: How much variance is explained (usually low for single predictors)

### What this means:
- A negative coefficient suggests mean reversion (big moves reverse)
- A positive coefficient suggests momentum (big moves continue)
- Low R-squared is normal - financial returns are hard to predict

### Distribution insights:
- Check skewness hints in the summaries
- Look at outlier counts (they can dominate results)
- Examine the histograms to see if assumptions hold